# Random Forest Experiment to Predict Nutrient Vector for Patients

In [ ]:
#Importing essential libraries
import pandas as pd
import numpy as np
#Libraries for machine learning
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, r2_score

#Setting seed for reproducibility
np.random.seed(55)


In [2]:
#Loading the excel file
df_2 = pd.read_csv("clean_pcos_data.csv")

In [ ]:
#Nutrient reccomender python logic

def nutrient_vector(patient_record):

    # Set a baseline daily reccomended intake of nutrients
    magnesium = 320.0 #milligrams
    fibre = 25.0 #grams
    PUFA = 12.0 #grams

    #Fibre reccomendation logic
    #High HOMA IR or high Fasting Glucose level can indicate insulin resistance
    #From high severity to mild using if and elif
    if patient_record['HOMA_IR'] > 5 or patient_record['Fasting_Glucose_mg_dL'] > 126:
        fibre += 10

    elif 3 <= patient_record['HOMA_IR'] <= 4.9 or 112.6 <= patient_record['Fasting_Glucose_mg_dL'] <= 125:
        fibre += 7.5 

    elif 2 <= patient_record['HOMA_IR'] <= 2.9 or 100 <= patient_record['Fasting_Glucose_mg_dL'] <= 112.5:
        fibre += 5
        
    elif patient_record['HOMA_IR'] > 1.9 or patient_record['Fasting_Glucose_mg_dL'] > 99:
        fibre += 2.5

    #Polyunsatured fats (PUFAs) reccommendation logic (as a replacement for Omega 3)
    #High triglycerides and the presence of severe acne can indicate high lipids and inflammation
    #Omega 3 can lower lipid levels and combat skin inflammation
    #From high severity to mild using if and elif
    if patient_record['Triglycerides_mg_dL'] > 249 or patient_record['Acne_Severity'] == 3:
        PUFA += 8
          
    elif 200 <= patient_record['Triglycerides_mg_dL'] <= 249 or patient_record['Acne_Severity'] == 2:
        PUFA += 4

    elif 150 <= patient_record['Triglycerides_mg_dL'] <= 199 or patient_record['Acne_Severity'] == 1:
        PUFA += 2
        

    #Magnesium reccomendation logic
    #Magnesium can support insulin resistance and hormonal imbalance
    #From high severity to mild using if and elif
    if patient_record['HOMA_IR'] > 5:
        if patient_record['PCOS_Diagnosis'] == 1:
            magnesium += 80
        else:
            magnesium += 70

    elif 3 <= patient_record['HOMA_IR'] <= 4.9:
        if patient_record['PCOS_Diagnosis'] == 1:
            magnesium += 50
        else:
            magnesium += 40
        
    elif 2 <= patient_record['HOMA_IR'] <= 2.9:
        if patient_record['PCOS_Diagnosis'] == 1:
            magnesium += 35
        else:
            magnesium += 25

    #Round the nutrient figures
    return [round(fibre, 1), round(PUFA, 1), round(magnesium, 1)]

#Testing the reccomender logic on first patient
patient_1 = df_2.iloc[0]
target_vector = nutrient_vector(patient_1)

print(f"Target Nutrient Vector for patient 1: {target_vector}")


Target Nutrient Vector for patient 1: [32.5, 20.0, 370.0]


In [4]:
#Running the nutrient vector logic across all rows in the PCOS dataset
df_2['Target_Nutrient_Vector'] = df_2.apply(nutrient_vector, axis=1)

print("Dataset with integrated target nutrient vector:")
print(df_2[['Fasting_Glucose_mg_dL', 'HOMA_IR', 'Triglycerides_mg_dL', 'Acne_Severity', 'LH_FSH_Ratio', 'Target_Nutrient_Vector']].head())

Dataset with integrated target nutrient vector:
   Fasting_Glucose_mg_dL   HOMA_IR  Triglycerides_mg_dL  Acne_Severity  \
0                   96.7  4.266738                  299              2   
1                  115.0  5.355309                  131              3   
2                   75.0  2.890741                  103              2   
3                  106.2  3.726178                  179              0   
4                   91.9  3.451356                  137              1   

   LH_FSH_Ratio Target_Nutrient_Vector  
0      1.551363    [32.5, 20.0, 370.0]  
1      2.939891    [35.0, 20.0, 390.0]  
2      1.298765    [30.0, 16.0, 345.0]  
3      1.401822    [32.5, 14.0, 360.0]  
4      1.666271    [32.5, 14.0, 360.0]  


In [ ]:
#Creating the nutrient vector for machine learning prediction
nutrient_list = []
for index, row in df_2.iterrows():
    vector = nutrient_vector(row)
    nutrient_list.append(vector)

X=df_2.drop(columns=['Target_Nutrient_Vector'])
Y=pd.DataFrame(nutrient_list, columns=['Target_Fibre', 'Target_PUFA', 'Target_Magnesium'])

#Splitting the data into training and test sets (80/20 split)
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.20, random_state=42)

#Seperate continuous features from ordinal/binary features for scaling
continuous_features = ['Menstrual_Cycle_Length_days', 'BMI', 'Fasting_Glucose_mg_dL',
                        'Fasting_Insulin_uIU_mL', 'HOMA_IR', 'LH_mIU_mL', 'FSH_mIU_mL', 'LH_FSH_Ratio',
                          'Total_Testosterone_ng_dL', 'Free_Testosterone_pg_mL', 'Total_Cholesterol_mg_dL', 'HDL_mg_dL', 'LDL_mg_dL', 'Triglycerides_mg_dL']
scaler = StandardScaler()
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

#Applying the scaler to the continuous columns only
X_train_scaled[continuous_features] = scaler.fit_transform(X_train[continuous_features])
X_test_scaled[continuous_features] = scaler.transform(X_test[continuous_features])

In [7]:
#Initialising a random forest model
rf_1 = RandomForestRegressor(n_estimators=100, random_state=64, max_depth=8)

#Predicting multiple continuous outputs
rf_nutrient_predictor = MultiOutputRegressor(rf_1)

#Training the model on the training subset
rf_nutrient_predictor.fit(X_train_scaled, Y_train)

#Making prediction on the test subset
Y_pred = rf_nutrient_predictor.predict(X_test_scaled)

#Creatinga dataframe for comparison of results
df_predictions = pd.DataFrame(Y_pred, columns=['Pred_Fibre', 'Pred_PUFA', 'Pred_Magnesium'], index=Y_test.index)

#Viewing the predictions
print(df_predictions.head())

     Pred_Fibre  Pred_PUFA  Pred_Magnesium
55         35.0      12.00           390.0
63         25.0      12.00           320.0
33         25.0      12.00           320.0
297        30.0      14.00           345.0
72         30.0      16.08           355.0


### Random Forest Model

In [ ]:
#Calculating the performance of the random forest predictions using metrics

for i, col in enumerate(Y.columns):
    mae = mean_absolute_error(Y_test.iloc[:, i], Y_pred[:, i])
    r2 = r2_score(Y_test.iloc[:, i], Y_pred[:, i])
    print(f" {col:<18} MAE: {mae:.2f} | R^2 Score: {r2:.2f}")

#The R square scores are very close to 100% suggesting severe overfitting.

 Target_Fibre       MAE: 0.18 | R^2 Score: 0.97
 Target_PUFA        MAE: 0.04 | R^2 Score: 0.99
 Target_Magnesium   MAE: 0.20 | R^2 Score: 1.00
